<a href="https://colab.research.google.com/github/BrunoCapron/ESQ724-fundamentos_aprendizado_maquina/blob/main/Otimizacao_Hiperparametros/exemplo_optuna_iris.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Exemplo de uso de Optuna para otimização de hiperparâmetros

Usaremos a base de dados IRIS para treinar uma floresta aleatória capaz de predizer a espécie de uma Iris a partir das dimensões das suas sépalas e pétalas. Os hiperparâmetros da floresta aleatória será otimizada usando a biblioteca Optuna.

In [3]:
#Instalamos a biblioteca optuna
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 8.4 MB/s eta 0:00:00


In [4]:
#Importamos a bibliotecas que serão usadas
import optuna
from sklearn.datasets import load_iris
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

Carregamos o conjunto de dados IRIS e dividimos os dados entre um um conjunto de treino e um conjunto de teste

In [5]:
iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

Define-se a função objetivo que o Optuna vai buscar maximizar

In [15]:
#a função objetivo vai treinar e avaliar o modelo com base nos hiperparâmetros sugeridos
def objective(trial):
  #sugere-se o espaço de busca dos hiperparâmetros para o Random Forest

  #n_estimators: número de árvores na floresta
  n_estimators = trial.suggest_int('n_estimators', 10, 200)

  #max_depth: profundidade máxima da árvore
  max_depth = trial.suggest_int('max_depth', 1, 32)

  #min_samples_split: número mínimo de amostras necessárias para dividir um nó
  min_samples_split = trial.suggest_float('min_samples_split', 0.1, 1.0) # Changed to suggest_float

  #criterion: função para medir a qualidade de uma divisão
  criterion = trial.suggest_categorical('criterion', ['gini', 'entropy'])

  #o modelo a ser treinado é criado
  model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, min_samples_split=min_samples_split, criterion=criterion,random_state=42)

  #o modelo é treinado e avaliado usando validação cruzada
  score = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy').mean()

  #funcao objetivo
  return score

Criamos e executamos o estudo (study)

In [16]:
study = optuna.create_study(direction = 'maximize') #informamos que queremos maximizar a função objetivo

print("Iniciando a otimização com Optuna...")
study.optimize(objective, n_trials=100) #executamos a otimização

print("Otimização concluída!")

[I 2025-11-20 16:48:28,854] A new study created in memory with name: no-name-78174203-5690-4302-868a-1141ff4ca901


Iniciando a otimização com Optuna...


[I 2025-11-20 16:48:29,752] Trial 0 finished with value: 0.9428571428571428 and parameters: {'n_estimators': 121, 'max_depth': 23, 'min_samples_split': 0.5100752703490551, 'criterion': 'entropy'}. Best is trial 0 with value: 0.9428571428571428.
[I 2025-11-20 16:48:29,963] Trial 1 finished with value: 0.9333333333333333 and parameters: {'n_estimators': 24, 'max_depth': 12, 'min_samples_split': 0.49285643345497165, 'criterion': 'gini'}. Best is trial 0 with value: 0.9428571428571428.
[I 2025-11-20 16:48:31,408] Trial 2 finished with value: 0.9333333333333333 and parameters: {'n_estimators': 195, 'max_depth': 5, 'min_samples_split': 0.26467055312415383, 'criterion': 'gini'}. Best is trial 0 with value: 0.9428571428571428.
[I 2025-11-20 16:48:32,219] Trial 3 finished with value: 0.3333333333333333 and parameters: {'n_estimators': 108, 'max_depth': 4, 'min_samples_split': 0.6994870385115344, 'criterion': 'entropy'}. Best is trial 0 with value: 0.9428571428571428.
[I 2025-11-20 16:48:33,819]

Otimização concluída!


Obtendo os melhores resultados:

In [17]:
print("\n--- Resultados da Otimização ---")
print(f"Melhor Acurácia de Validação Cruzada: {study.best_value:.4f}")
print("Melhores Hiperparâmetros:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")


--- Resultados da Otimização ---
Melhor Acurácia de Validação Cruzada: 0.9524
Melhores Hiperparâmetros:
  n_estimators: 113
  max_depth: 8
  min_samples_split: 0.65222285804398
  criterion: gini


Treinamos o modelo com os melhores hiperparâmetros

In [18]:
best_params = study.best_params
final_model = RandomForestClassifier(**best_params, random_state=42)
final_model.fit(X_train, y_train)

RandomForestClassifier(max_depth=8, min_samples_split=0.65222285804398,
                       n_estimators=113, random_state=42)

Avaliamos o modelo em cima do conjunto de teste

In [19]:
y_pred = final_model.predict(X_test)
final_accuracy = accuracy_score(y_test, y_pred)

print(f"\nAcurácia do Modelo Final no Conjunto de Teste: {final_accuracy:.4f}")


Acurácia do Modelo Final no Conjunto de Teste: 0.9556
